# Project's Baseline
This notebook contains the baseline of the Deep Learning Project. The pipeline is the following one:
1. Setup: download CLIP and data
2. Visual features extraction: use CLIP image encoder to obatin visual features and store them (frozen visual features)
3. Text features: take the queries and encode them with the CLIP text encoder.
4. Latent space arithmetic and model predictions: given a reference image (.js file) and a query obatin the target vith v_target = v_ref +/- query; compute the cosine similarity between the target and the other images (the k data images with high cossim are the predictions)
5. Model evaluation: compute evaluation metrics.

---
## 1.Setup

Import the code dependencies.

In [ ]:
from google.colab import drive
import torch
from pathlib import Path
from torchvision.datasets import CelebA
import numpy as np
from PIL import Image
import torchvision
import torch.nn.functional as F
import json
from torch.utils.data import DataLoader
import shutil

### Dataset setup

Unzip the caleba data into the runtime's loacl SSD and extract caleba data.

In [ ]:
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
!mkdir /content/datasets

In [ ]:
# unzip the dataset in the runtime's local SSD
!unzip -q /content/drive/MyDrive/deep_learning/data/celeba.zip -d /content/datasets/

In [ ]:
celeba = CelebA(root=Path("/content/datasets"), split="test", download=False)


In [ ]:
# DEBUG: analyze celeba dataset
print(celeba)
print(type(celeba))
print(celeba.attr_names)
print(type(celeba[0][1]))
print(celeba[0])
print(type(celeba.filename[0]))

Dataset CelebA
    Number of datapoints: 19962
    Root location: /content/datasets
    Target type: ['attr']
    Split: test
<class 'torchvision.datasets.celeba.CelebA'>
['5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young', '']
<class 'torch.Tensor'>


str

### CLIP download


Install CLIP and analyze the model.

In [ ]:
!pip install -q huggingface_hub transformers

In [ ]:
from transformers import CLIPModel, CLIPProcessor

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

model = model.cuda().eval() # move to the GPU

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
# DEBUG: analyze CLIP model
print("Model type:", model.config.model_type)
print("Projection dim:", model.config.projection_dim)
print("Text hidden size:", model.config.text_config.hidden_size)
print("Vision hidden size:", model.config.vision_config.hidden_size)
print("Vision image size:", model.config.vision_config.image_size)
print("Vision patch size:", model.config.vision_config.patch_size)
print("Vocabulary size:", model.config.text_config.vocab_size)
print(
    "Max position embeddings:", model.config.text_config.max_position_embeddings
)

Model type: clip
Projection dim: 512
Text hidden size: 512
Vision hidden size: 768
Vision image size: 224
Vision patch size: 32
Vocabulary size: 49408
Max position embeddings: 77


---
## 2. Visual features extraction

encode_images function is used to extract the visual features of CelebA data trough the visual encoder of CLIP. This function processes one image at time and not the entire list of images because the RAM of Colab is limited. For each image, the function stores its visual features (a tensor) in a file ".pt" in the path passed in input.
> **Warning**
> This section must be **executed if and only if the visual features of CelebA data have not been extracted yet**.

In [ ]:
def encode_images(dataset, output_path):
  """Extract visual features from celeba data and froze them.

  Arg:
    dataset: celeba data
    output_path: path where to store the visual features
  """
  for i in range(len(celeba)):
      # obtain tensor of the PIL image and move it to cuda
      # processor returns a BatchFeature, we need its pixel_values
      image_t = processor(
          images=celeba[i][0],
          return_tensors="pt",
          padding = True
      )
      image_t_p = image_t.pixel_values.to("cuda") # Move pixel values to CUDA

      # encode the images with CLIP (latent space)
      with torch.no_grad():
          image_z_output = model.get_image_features(pixel_values=image_t_p)

      image_z = image_z_output.pooler_output # the tensor
      # normalize the fetaures
      image_z = image_z / image_z.norm(dim=-1, keepdim=True)

      # save image encoded features to a file
      image_output_path = output_path / f"{i}.pt"
      torch.save(image_z.cpu(), image_output_path)

In [ ]:
!mkdir /content/frozen_data

In [ ]:
output_path = Path("/content/frozen_data")
encode_images(celeba, output_path)

In [ ]:
# create a zip of the visual features
shutil.make_archive("frozen_data", "zip", "/content/frozen_data")

'/content/frozen_data.zip'

### Unzip visual features


In [ ]:
!mkdir /content/frozen_data

In [ ]:
!unzip -q /content/drive/MyDrive/deep_learning/data/frozen_data.zip -d /content/datasets/frozen_data

---
## 3. Text features

In [ ]:
def encode_queries(queries):
  """Extrcat text features from queries texts.

  Arg:
    queries: list of queries in the .js file

  Return:
    text_z: list of (normalized) text features
  """
  # tokenize the input queries
  text_t = processor(text=queries, return_tensors="pt", padding=True)

  # input tensors to cuda
  text_ids = text_t['input_ids'].to("cuda")
  # attention mask to cuda
  attention_mask = text_t['attention_mask'].to("cuda")

  # encode the text with CLIP (latent space)
  with torch.no_grad():
      text_z_output = model.get_text_features(
          input_ids=text_ids,
          attention_mask=attention_mask
      )

  text_z = text_z_output.pooler_output # the tensor
  # normalize the features
  text_z = text_z / text_z.norm(dim=-1, keepdim=True)

  return text_z

In [ ]:
def get_annotations(annotations_path):
  """Extract queries and groundtruth annotations from .js file.

  Arg:
    annotations_path: path to the .js file

  Return:
    annotations: list of queries and groundtruth annotations
  """
  with open(annotations_path, "r") as f:
    annotations = json.load(f)

  return annotations

In [ ]:
def get_unsigned_queries(annotations):
  """Creates a list of unsigned queries (ex: +Smiling becomes Smiling)

  Arg:
    annotations: list of queries and groundtruth annotations

  Return:
    queries_unsigned: list of unsigned queries
  """
  queries = [a["query"] for a in annotations]

  queries_unsigned = []
  for query in queries:
    # if a query contains multiple features, divide them
    # and remove sign
    if ',' in query:
      multiple_q = [q.strip() for q in query.split(",")]
      multiple_q = [q[1:] for q in multiple_q]
      queries_unsigned += multiple_q
    else:
      # remove sign (first character)
      queries_unsigned.append(query[1:])

  return list(dict.fromkeys(queries_unsigned))


In [ ]:
annotations_path = Path(
    "/content/drive/MyDrive/deep_learning/data/celeba_evaluation.json"
)
# get annotations (queries and gt)
annotations = get_annotations(annotations_path)
# get queries
queries_unsigned = get_unsigned_queries(annotations)
# obtain text features of the queries
text_features = encode_queries(queries_unsigned)
# store mapping between text features and unsigned queries
texts = dict(zip(queries_unsigned, text_features))

In [ ]:
# DEBUG: analyze shape of images features and text features
visual_path = Path("/content/datasets/frozen_data")
filename = f"{1}.pt"
file_path = visual_path / filename
data = torch.load(file_path, map_location="cpu")
print(f"Text features shape: {text_features[0].shape}")
print(f"Image features shape: {data.shape}")



Text features shape: torch.Size([512])
Image features shape: torch.Size([1, 512])


---
## 4. Latent space arithmetic and model predicitions

All this code is bad, but it is like this to avoid exhausting Colab resources (principally RAM). If we want to use it in the final project (even if I do not thing so because it is the baseline) we need to write it better.

In [ ]:
def cosine_similarity(image_query_z, image_z):
  """Computes cosine similarity between tewo images.

  Arg:
    image_query_z: (normalized) image features modified with the query texts
    image_z: (normalized) image features

  Return:
    cosine similarity between the two images
  """
  return F.cosine_similarity(image_query_z.unsqueeze(1), image_z.unsqueeze(0), dim=-1).cpu()

In [ ]:
def modify_target(visual_path, target_idx, texts, query):
  """Apply query to the target image (latent space arithmetic)

  Arg:
    visual_path: path where images features are contained
    target_idx: index of the target image in celeba
    texts: dictionary of unsigned queries and text features
    query: query to be applied

  Return:
    target_tensor: tensor of the target image modified with the query
  """
  # extract visual features of the specific target
  filename = f"{target_idx}.pt"
  file_path = visual_path / filename
  target_tensor = torch.load(file_path, map_location="cpu").to("cuda")


  # perform naive arithmetic operations in the latent space
  multiple_q = [q.strip() for q in query.split(",")]
  for q in multiple_q:
      if '+' in q:
          target_tensor = target_tensor + texts[q[1:]].unsqueeze(0)
      else:
          target_tensor = target_tensor - texts[q[1:]].unsqueeze(0)

  target_tensor = target_tensor / target_tensor.norm(dim=-1, keepdim=True)

  return target_tensor

In [ ]:
def get_predictions(modified_target, k, data_path):
  """Find the predictions based on cosine similarity.

  Arg:
    modified_target: tensor of the target image modified with the query
    k: the cutoff for top-K evaluation (e.g., 1, 5, 10)
    data_path: path where images features are contained

  Return:
    predictions: dictionary of the first k predictions
  """
  images_sim_cos = {}
  for pt_file in data_path.glob("*.pt"):
      tensor = torch.load(pt_file).to("cuda")
      idx = int(pt_file.name.replace(".pt", ""))
      images_sim_cos[idx] = cosine_similarity(modified_target, tensor)

  predictions = sorted(
    images_sim_cos.items(),
    key=lambda x: x[1],
    reverse=True
  )
  predictions = dict(predictions[:k])

  return predictions


In [ ]:
# DEBUG: first target image for query +Smiling
target_images_ids = list(annotations[0]["ground_truth"].keys())
directory = Path("/content/datasets/frozen_data")
# create the v_target = v_ref + query
modified_target = modify_target(
    directory,
    target_images_ids[0],
    texts,
    annotations[0]["query"]
)
# find the first 10 predictions ordered by cosine similarity
predictions = get_predictions(modified_target, 10, directory)
print(predictions)


torch.Size([1, 512])
{13: tensor([[0.7854]]), 13877: tensor([[0.6794]]), 16927: tensor([[0.6782]]), 10996: tensor([[0.6736]]), 8348: tensor([[0.6733]]), 14089: tensor([[0.6732]]), 3874: tensor([[0.6718]]), 4868: tensor([[0.6711]]), 15270: tensor([[0.6710]]), 17732: tensor([[0.6697]])}


---
## 5. Model evaluation

Function to evaluate (recall and precision) the final model. This function was given by the professor in the project's skeleton.

In [ ]:
def evaluate_retrieval(
    retrieved_indices: list[int],
    ground_truth_indices: list[int],
    k: int
):
    """
    Evaluate the retrieval performance for a single source image.

    Args:
    ----
        retrieved_indices: list of image IDs predicted by the model,
            ordered by similarity (descending).
        ground_truth_indices: list of valid target IDs from the benchmark JSON.
        k: the cutoff for top-K evaluation (e.g., 1, 5, 10).

    Return:
    ------
        A dictionary containing Recall@K and Precision@K.

    """
    # Isolate the top K predictions
    top_k_retrieved = retrieved_indices[:k]

    # Calculate the intersection between predictions and ground truth
    hits = set(top_k_retrieved).intersection(set(ground_truth_indices))
    num_hits = len(hits)

    # Metrics calculations
    # Recall@K (Hit Rate): 1 if at least one match is found, 0 otherwise
    recall_at_k = 1 if num_hits > 0 else 0

    # Precision@K: Fraction of top K predictions that are correct
    precision_at_k = num_hits / k

    return {
        f"Recall@{k}": recall_at_k,
        f"Precision@{k}": precision_at_k
    }

In [ ]:
# DEBUG: first target image for query +Smiling
evaluate_retrieval(
    list(predictions.keys()),
    list(annotations[0]["ground_truth"][target_images_ids[0]]),
    10
)

{'Recall@10': 0, 'Precision@10': 0.0}

In [ ]:
# DEBUG
def test_query(query_id, k, n_images, directory):
  """Test the model with a specific query.

  Arg:
    query_id: index of the query in the .js file
    k: the cutoff for top-K evaluation (e.g., 1, 5, 10)
    n_images: number of reference images to test the model
    directory: path where images features are contained
  """
  target_images_ids = list(annotations[query_id]["ground_truth"].keys())
  results = []
  for id in target_images_ids[:n_images]:
    modified_target = modify_target(
        directory,
        id,
        texts,
        annotations[query_id]["query"]
    )
    predictions = get_predictions(modified_target, k, directory)
    eval = evaluate_retrieval(
        list(predictions.keys()),
        list(annotations[query_id]["ground_truth"][id]),
        k
    )
    results.append(eval)

  print(results)


In [ ]:
# DEBUG
directory = Path("/content/datasets/frozen_data")
# TEST 1: first 3 images for query +Smiling
# test_query(0, 10, 3, directory)
# TEST 2: first 5 images for query +Eyeglasses
# test_query(1, 10, 5, directory)
